# Student Performance Prediction
## A comparative study of ML models on the OULAD Dataset
### What this project does (A Big Picture)
- We are using OULAD dataset which provides us with ~30,000 university students - their demographics (age, gender, region, etc) and their scores on assessments.
- Our goal: can we predict whether a student will pass or fail?
- We are going to train 3 machine learning models and compare them:
  - Logistic Regression
  - XGBoost
  - Neural Network (MLP)

## Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

ModuleNotFoundError: No module named 'xgboost'

## Loading Data

In [ ]:
info = pd.read_csv("studentInfo.csv", na_values="?")
assess = pd.read_csv("studentAssessment.csv", na_values="?")

In [ ]:
info.head()

In [ ]:
assess.head()

In [ ]:
print(info.shape)
print(assess.shape)

## Cleaning Data

In [ ]:
# convert values in scores column into numbers, if the value is other than number (e.g. N/A or ?), put NaN instead
assess['score'] = pd.to_numeric(assess['score'], errors='coerce')

# delete any row where the score column is NaN
assess = assess.dropna(subset=['score'])

In [ ]:
print(assess.shape)

## Create Target Variable

- The `final_result` column in the info dataset has 4 different values viz Pass, Distinction, Fail, and Withdrawn
- To simplify the classification, we want to convert that one column into 4 different columns so, we can use 0 and 1 to label whether Pass is yes or no or Distinction is yes or no.
- This is what the model will be trained on to predict the outcome

In [ ]:
info['passed'] = info['final_result'].map({
    'Pass': 1,
    'Distinction': 1,
    'Fail': 0,
    'Withdrawn': 0,
})
info.shape

In [ ]:
# remove rows from the dataset where the value of 'passed' column is empty
info = info.dropna(subset=['passed'])
info.shape

In [ ]:
# converting decimal values to integers in passed column using astype()
info['passed'] = info['passed'].astype(int)

In [ ]:
# count the number of students passed/failed
pass_counts = info['passed'].value_counts()
print(f"   Pass (1): {pass_counts[1]} students  ({pass_counts[1]/len(info)*100:.1f}%)")
print(f"   Fail (0): {pass_counts[0]} students  ({pass_counts[0]/len(info)*100:.1f}%)")

## Summarizing Scores for Each Student

- The assessment file has multiple rows per student (one per quiz)
- We need one row per student to combine with their profile
- So, all of the quiz rows will be collapsed into number of quizzes, score average, and standard deviation to understand their consistency

In [ ]:
# Using groupby() to combine all the rows that belong to the same student
# and agg() to calculate average, standard deviation, and total number of quizzes
student_features = assess.groupby('id_student').agg(
    avg_score = ('score', 'mean'),
    score_std = ('score', 'std'),
    num_assessments = ('score', 'count')
).reset_index()

student_features

## Merging Datasets
- Merging the info and student_features tables together

In [ ]:
# Merging the two tables using the student ID via merge()
# `how='inner'` means similar inner join in SQL i.e.
# keep the rows that exist in both tables
df = pd.merge(info, student_features, on='id_student', how='inner')
df.head()

In [ ]:
df.shape


In [ ]:
df.info()

In [ ]:
df['num_assessments'].min()

In [ ]:
df['num_assessments'].max()

In [ ]:
df['studied_credits'].min()

In [ ]:
df['studied_credits'].max()

## Visualization 01 - Class Distribution

### Chart 01: Pass vs. Fail Count
- How many students passed vs. failed?
- A bar chart:
  - red bar = failed
  - green bar = passed
 
### Chart 02: Average Score
- Do students who pass score higher on average?
- Two overlapping histograms - one for students who passed and one for students who failed
- If the two histograms overlap a lot, scores alone do not tell us much
- If they are far apart, scores are a strong predictor

### Chart 03: Credits vs Pass Rate
- Does taking more credits affect passing?
- Grouping students by how many credits they studied, then calculating pass rate for each group.

In [ ]:
# Make a canvas with 3 charts side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('OULAD Student Performance — Exploratory Analysis', fontsize=16, fontweight='bold')

# -------------------------------
# Chart 1: Pass vs Fail Count
# -------------------------------
sns.countplot(x='passed', data=df, hue='passed', palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_title('Pass vs Fail — How many students?')
axes[0].set_xlabel('0 = Failed,  1 = Passed')
axes[0].set_ylabel('Number of Students')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Fail (0)', 'Pass (1)'])

for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# -------------------------------
# Chart 2: Average Score
# -------------------------------
fail_scores = df[df['passed'] == 0]['avg_score']
pass_scores = df[df['passed'] == 1]['avg_score']

axes[1].hist(fail_scores, bins=30, alpha=0.7, color='#e74c3c', label='Fail')
axes[1].hist(pass_scores, bins=30, alpha=0.7, color='#2ecc71', label='Pass')
axes[1].set_title('Do Pass students score higher?')
axes[1].set_xlabel('Average Quiz Score')
axes[1].set_ylabel('Number of Students')
axes[1].legend()

# -------------------------------
# Chart 3: Credits vs Pass Rate
# -------------------------------
credit_pass = df.groupby('studied_credits')['passed'].mean().reset_index()
axes[2].bar(credit_pass['studied_credits'].astype(str), credit_pass['passed'],
            color='#3498db', alpha=0.8)
axes[2].set_title('Does number of credits affect passing?')
axes[2].set_xlabel('Studied Credits')
axes[2].set_ylabel('Pass Rate (0 to 1)')
axes[2].tick_params(axis='x', rotation=45)

# Show charts
plt.tight_layout()
plt.show()

## Encoding Categorical Variables
- Converting every text column into 0s and 1s using one-hot encoding

In [ ]:
exclude_cols = ["id_student", "passed", "final_result", "avg_score"]

# finding all the columns that contain text excluding exclude_cols
categorical_cols = [col for col in df.select_dtypes(include=["object"]).columns if col not in exclude_cols]

categorical_cols

In [ ]:
# get_dummies() actually performs the one-hot encoding
# drop_first=True removes one duplicate category from each column
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
df_encoded

## Defining Features (X) and Target (Y)
- We removed:
  - `id_student`: just an ID number, not useful for prediction
  - `passed`: this is the target variable, not a feature
  - `final_result`: this is the raw version of the target variable
  - `avg_score` - this is removed because it is also directly linked to passing, in order to make the prediction more challenging and realistic

In [ ]:
cols_to_drop = ['id_student', 'passed', 'avg_score'] + \
               [col for col in df_encoded.columns if 'final_result' in col]

In [ ]:
X = df_encoded.drop(columns=cols_to_drop)

In [ ]:
y = df_encoded['passed']

## Train/Test Split
- Training set 80%
- Testing set 20%

In [ ]:
# random_state=42 makes sure that the split is always the same every time we run the code
# stratify=y makes sure that both train/test sets have the same ratio of pass/fail students
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape[0]

In [ ]:
X_test.shape[0]

## Feature Scaling/Data Normalization

- Some features have very different ranges:
  - `num_assessments` ranges from 1 to 28
  - `studied_credits` ranges from 30 to 630
 
- Logistic Regression and Neural Networks are sensitive to this.
- They give more weight to larger numbers even if they are not more important.

- We are using StandardScaler to normalize every feature
  - mean = 0
  - standard deviation = 1


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Fill any remaining missing values with 0
X_train_scaled = np.nan_to_num(X_train_scaled)
X_test_scaled = np.nan_to_num(X_test_scaled)

## Helper Function to Evaluate a Model
- This function takes a trained model and test data, makes predictions, prints all the metrics, and returns the scores.

In [ ]:
def evaluate_model(name, model, X_test_data, y_test_data):
    """
    Evaluates a trained model and prints its performance metrics.
    
    Parameters:
        name        - Name of the model (for display)
        model       - The trained sklearn model object
        X_test_data - Test features
        y_test_data - True test labels
    
    Returns:
        dict with accuracy, precision, recall, f1
    """
    # make predictions on the test set
    # .predict() outputs 0 or 1 for each statement
    y_pred = model.predict(X_test_data)

    # calculate metrics
    acc = accuracy_score(y_test_data, y_pred)
    prec = precision_score(y_test_data, y_pred, zero_division=0)
    rec = recall_score(y_test_data, y_pred, zero_division=0)
    f1 = f1_score(y_test_data, y_pred, zero_division=0)

    print(name)
    print("")
    print("Accuracy:   " + str(round(acc, 4)))
    print("Precision:  " + str(round(prec, 4)))
    print("Recall:     " + str(round(rec, 4)))
    print("F1 Score:   " + str(round(f1, 4)))
    print("")
    print(classification_report(y_test_data, y_pred, target_names=['Fail', 'Pass']))

    return {'name': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'y_pred': y_pred}
    

## Model 01: Logistic Regression
- Logistic Regression draws a straight line that separates Pass students from Fail students.
- It outputs a probability (0 to 1) of passing then:
  - probablity >= 0.5 predicts Pass (1)
  - probability < 0.5 predicts Fail (0)

In [ ]:
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_results = evaluate_model("MODEL 01: Logistic Regression", lr_model, X_test_scaled, y_test)

## Model 02: XGBoost
- more info

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

# train model
xgb_model.fit(X_train, y_train)

# evaluate
xgb_results = evaluate_model(
    "MODEL 02: XGBoost",
    xgb_model,
    X_test,
    y_test
)

## Model 03: Neural Network (MLP)
- more info

In [ ]:
# code

## Visualization 02 - Model Comparison Bar Chart

In [ ]:
results = [lr_results, xgb_results]

summary = pd.DataFrame(results)[["name", "accuracy", "precision", "recall", "f1"]]

# print table
print(summary)

# create bar chart
summary.set_index("name")[["accuracy", "f1"]].plot(kind="bar", figsize=(8,5))

plt.title("Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

## Visualization 03 - Confusion Matrices
- more info

In [ ]:
# code

## Visualization 04 - XGBoost Feature Importance
- more info

In [ ]:
# code

## Final Summary Table
- more info

In [ ]:
# code